Notebook experimentation for the Document Pretrained Transformer (DPT). The DPT is a special kind of transformer that is trained on understanding document layout, complex document structures and token prediction. In my version of the DPT, I train the model using the Masked Language Model (MLM) approach where a huge dataset corpus like the wikitext-103 which contains many articles, books, etc and has the corpus size of 103 million tokens. Each article is treated as a document and the tokens are given individual bounding boxes with 2D bounding box coordinates. This teaches the model granular layout structure by learning how each token can be predicted by its surrounding tokens and spatial embeddings.

### Exceptions Class

In [ ]:
class ProjectException(Exception):
    """Base exception for the Document Layout Transformer project.

    Args:
        message (str): Human-readable error description.
        error_detail (any, optional): Additional technical detail (e.g., sys.exc_info()).
    """
    def __init__(self, message: str, error_detail=None):
        super().__init__(message)
        self.error_detail = error_detail

    def __str__(self):
        return f"[ProjectException] {super().__str__()}" + \
               (f" | Detail: {self.error_detail}" if self.error_detail else "")

### Custom Logging

In [ ]:
import logging
import sys

def setup_logger(name: str, level=logging.INFO):
    """Sets up a custom logger with a specific name, level, and format."""
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Prevent adding multiple handlers if the logger is already configured
    if not logger.handlers:
        # Create a console handler
        handler = logging.StreamHandler(sys.stdout)

        # Define the log format
        formatter = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger

# Example Usage:
# Get a logger instance for your module/script
logger = setup_logger(__name__)

# Test different logging levels
logger.debug("This is a debug message.")
logger.info("This is an info message.")
logger.warning("This is a warning message.")
logger.error("This is an error message.")
logger.critical("This is a critical message.")

# You can also change the level dynamically
# logger.setLevel(logging.DEBUG)
# logger.debug("This debug message will now be visible.")

### Data Acquisition

In [ ]:
import os
import shutil
from pathlib import Path
import kagglehub

# Define base directories
CONTENT_DIR = Path("/content")
DATA_DIR = CONTENT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

# Datasets to download (name on Kagglehub, expected local folder name after download)
DATASETS = {
    "wikitext-103": "vadimkurochkin/wikitext-103",

    "funsd": "aravindram11/funsdform-understanding-noisy-scanned-documents"
}

#checks if the dataset exists and if there is anything in the dataset pathg
def is_dataset_present(dataset_dir: Path) -> bool:
    """Check if dataset directory exists and contains at least one file."""
    return dataset_dir.exists() and any(dataset_dir.iterdir())


for local_name, ds_name in DATASETS.items():
    local_path = DATA_DIR / local_name
    if is_dataset_present(local_path):
        print(f"✅ {ds_name} already cached at {local_path}")
    else:
        print(f"⬇️ Downloading {ds_name} from Kagglehub...")
        try:
            # kagglehub returns path to downloaded files
            downloaded_path = kagglehub.dataset_download(ds_name)
            # Move to our data directory (or symlink)
            shutil.copytree(downloaded_path, local_path)
            print(f"✅ Downloaded {ds_name} to {local_path}")
        except Exception as e:
            raise ProjectException(f"Failed to download dataset {ds_name}", error_detail=e)

### EDA for wikitext-103

In [ ]:
train_file = None

likely_files = ["wiki.train.tokens", "train.tokens", "wiki.train.raw"]
for pattern in likely_files:
    matches = list((DATA_DIR / "wikitext-103").rglob(pattern))
    if matches:
        train_file = matches[0] #pick the first file->wiki.train.tokens
        break
if train_file is None:
    raise ProjectException("Could not find wiki.train.tokens")

print(f"File: {train_file}")

size_of_file = train_file.stat().st_size // (1024 * 1024)
print(f"Size of file: {size_of_file:,} MB\n")

# Read and examine first 30 non‑empty lines and detect blank lines
with open(train_file, "r", encoding="utf-8") as f:
    for i, raw_line in enumerate(f):
        line = raw_line.rstrip("\n\r")
        # Print first 20 lines regardless
        if i < 30:
            print(f"Line {i:4d} | len={len(line):4d} | content={repr(line[:120])}") #truncate line to 120 chars
        # Check for blank lines in first 200 lines
        if i < 200 and line == "":
            print(f">>> Blank line found at index {i}")
        if i >= 200:
            break

In [ ]:
import os
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

# Locate the downloaded wiki dataset
wiki_dir = DATA_DIR / "wikitext-103"
if not wiki_dir.exists():
    raise ProjectException(f"WikiText-103 directory not found at {wiki_dir}")

# Find the training file
train_file = None
for pattern in likely_files:
    matches = list(wiki_dir.rglob(pattern))
    if matches:
        train_file = matches[0]
        break
if train_file is None:
    raise ProjectException("Could not find wiki.train.tokens (or similar)")

print(f"Reading {train_file}...")

# Read lines, group into articles.
# Article boundaries are lines that contain only whitespace (e.g., a single space).
articles = []
current_lines = []
with open(train_file, "r", encoding="utf-8") as f:
    for raw_line in f:
        line = raw_line.rstrip("\n\r")
        # If line is empty or only whitespace → separator
        if line.strip() == "":
            if current_lines:
                # Join the collected lines with newlines to keep original structure
                articles.append("\n".join(current_lines))
                current_lines = []
        else:
            current_lines.append(line)
    # Last article
    if current_lines:
        articles.append("\n".join(current_lines))

num_articles = len(articles)
print(f"Number of articles: {num_articles}")

# Compute article lengths (whitespace‑split tokens)
token_lengths = [len(art.split()) for art in articles]
total_tokens = sum(token_lengths)
avg_len = total_tokens / num_articles if num_articles else 0
print(f"Total tokens (whitespace split): {total_tokens:,}")
print(f"Average article length: {avg_len:,.1f} tokens")
print(f"Max article length: {max(token_lengths):,} tokens")
print(f"Min article length: {min(token_lengths):,} tokens")

# Plot histogram
plt.figure(figsize=(10,4))
plt.hist(token_lengths, bins=50, edgecolor='black')
plt.title("WikiText-103 Article Length Distribution (tokens)")
plt.xlabel("Tokens per article")
plt.ylabel("Frequency")
plt.show()

# Show a sample article (first 3)
print("\n--- First 3 articles (truncated) ---")
for i, art in enumerate(articles[:3]):
    print(f"\nArticle {i+1} ({len(art.split())} tokens):")
    print(art[:500] + "..." if len(art) > 500 else art)
    print("-" * 50)

### Visualization of sythentic 2d bounding boxes for a sample article

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Pick a short article for clear visualisation
sample_article = None
for art in articles:
    if 100 < len(art.split()) < 300:   # not too short, not too long
        sample_article = art
        break #break becuase there is going to be a lot of articles that fall within this criteria
if sample_article is None:
    sample_article = articles[0]   # fallback

print(f"Sample article length (whitespace tokens): {len(sample_article.split())}")

# Split article into lines using newline (the article may contain actual '\n')
lines = sample_article.split('\n')
# Filter out empty lines
lines = [line.strip() for line in lines if line.strip()]
num_lines = len(lines)

# For each line, get tokens (already whitespace separated)
line_tokens = [line.split() for line in lines]

max_tokens_per_line = max(len(tokens) for tokens in line_tokens)
print(f"Number of lines: {num_lines}, Max tokens per line: {max_tokens_per_line}")

# Generate normalised bounding boxes
boxes = []   # list of (x0, y0, x1, y1) per token
texts = []   # the actual token string
for line_idx, tokens in enumerate(line_tokens):
    for token_idx, token in enumerate(tokens):
        x0 = token_idx / max_tokens_per_line
        x1 = (token_idx + 1) / max_tokens_per_line
        y0 = line_idx / num_lines
        y1 = (line_idx + 1) / num_lines
        boxes.append((x0, y0, x1, y1))
        texts.append(token)

# Plotting the tokens with the generic bounding boxes around it
fig, ax = plt.subplots(figsize=(max(12, max_tokens_per_line/2), max(6, num_lines/2)))

# Draw rectangles and text
for bbox, token in zip(boxes, texts):
    x0, y0, x1, y1 = bbox
    width = x1 - x0
    height = y1 - y0
    rect = patches.Rectangle((x0, y0), width, height,
                             linewidth=1, edgecolor='blue', facecolor='lightblue', alpha=0.4)
    ax.add_patch(rect)
    # Place token text (truncate if too long)
    display_token = token[:15] + ('…' if len(token) > 15 else '')
    ax.text(x0 + width/2, y0 + height/2, display_token,
            ha='center', va='center', fontsize=8, clip_on=True)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks(np.linspace(0, 1, max_tokens_per_line+1))
ax.set_yticks(np.linspace(0, 1, num_lines+1))
ax.grid(True, linestyle='--', alpha=0.5)
ax.set_title("Synthetic Layout Grid (WikiText-103 sample)", fontsize=14)
ax.set_xlabel("Normalised X (token index / max tokens)")
ax.set_ylabel("Normalised Y (line index / total lines)")
ax.invert_yaxis()   # so line 0 is at top
plt.tight_layout()
plt.show()

### EDA for the FUNSD

In [ ]:
##this is how the annotation for each document looks like

# FUNSD structure:
# {"form": [
#   {
#     "id": ..., "text": "field text?", "label": "question"/"answer"/"header"/"other",
#     "words": [
#       {"text": "word", "box": [x0,y0,x1,y1]},
#       ...
#     ],
#     "linking": [...], ...
#   },
#   ...
# ]}
# We need to assign each word the label of its parent form item.


In [ ]:
import json
import cv2
import numpy as np
import random
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Locate FUNSD directory
funsd_dir = DATA_DIR / "funsd"
if not funsd_dir.exists():
    raise ProjectException(f"FUNSD directory not found at {funsd_dir}")

# Find all PNG images recursively
image_paths = list(funsd_dir.rglob("*.png"))
if not image_paths:
    raise ProjectException("No PNG images found in FUNSD dataset")
print(f"Found {len(image_paths)} document images")

# Pick random image from the directory
sample_img_path = random.choice(image_paths) #random.choice selects one at random
stem = sample_img_path.stem #get the file name(stem-> usually what comes before .png)

#find its corresponding json annotations
found_jsons = list(funsd_dir.rglob(f"{stem}.json"))
if not found_jsons:
    raise ProjectException(f"No annotation JSON found for {sample_img_path}")
sample_json_path = found_jsons[0]

print(f"Sample image: {sample_img_path.name}")
print(f"Sample annotation: {sample_json_path.name}")

# Load image with cv2
img = cv2.imread(str(sample_img_path))
if img is None:
    raise ProjectException(f"Failed to load image {sample_img_path}")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
height, width, _ = img.shape
print(f"Image dimensions: {width}x{height}")

# Load annotation
with open(sample_json_path, "r", encoding="utf-8") as f:
    annot = json.load(f)


all_words = []

for form_item in annot.get("form", []):
    label = form_item.get("label", "other") #either the particular field has a label or if not then other
    for word_info in form_item.get("words", []):
        word_info = word_info.copy()  # avoid modifying original
        word_info["label"] = label
        all_words.append(word_info)

print(f"Total annotated words: {len(all_words)}")

# Label distribution
label_counter = Counter(word["label"] for word in all_words)
print("\nLabel distribution:")
for label, count in label_counter.items():
    print(f"  {label}: {count}")

# Visualize with bounding boxes and labels
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
colors = {"question": "blue", "answer": "green", "header": "orange", "other": "black"}
print(f"Colors mapping: {colors}")

for word in all_words:
    box = word["box"]  # [x0, y0, x1, y1]
    label = word["label"]
    color = colors.get(label, "black")
    rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                             linewidth=1, edgecolor=color, facecolor='none')
    ax.add_patch(rect)

# Add label text for first 30 words (to avoid clutter)
for word in all_words[:30]:
    box = word["box"]
    ax.text(box[0], box[1] - 5, word["label"], fontsize=7, color='red',
            bbox=dict(facecolor='white', alpha=0.7))

ax.set_title("FUNSD Sample with Entity Labels", fontsize=14)
plt.tight_layout()
plt.show()

# Aggregate statistics over first 50 documents
all_doc_stats = []
for img_p in list(funsd_dir.rglob("*.png"))[:50]:
    json_p = img_p.with_suffix(".json")
    if not json_p.exists():
        found = list(funsd_dir.rglob(f"{img_p.stem}.json"))
        if found:
            json_p = found[0]
        else:
            continue
    with open(json_p, "r") as f:
        doc_annot = json.load(f)
    word_count = 0
    for form_item in doc_annot.get("form", []):
        word_count += len(form_item.get("words", []))
    if word_count > 0:
        all_doc_stats.append(word_count)

if all_doc_stats:
    print(f"\nAggregate stats on {len(all_doc_stats)} documents:")
    print(f"  Avg words per doc: {np.mean(all_doc_stats):.1f}, max: {max(all_doc_stats)}")

plt.figure(figsize=(8, 4))
plt.hist(all_doc_stats, bins=30, edgecolor='black')
plt.title("Word Count per Form (FUNSD)")
plt.xlabel("Number of words")
plt.ylabel("Frequency")
plt.show()

### Tokenizer(BERT-Based-Uncased)

In [ ]:
from transformers import AutoTokenizer
import torch

# defining TokenizerWrapper
class TokenizerWrapper:
    def __init__(self, pretrained_name="bert-base-uncased",
                 max_length=512):
        """Wrapper for the tokenizer,
        Args:
            pretrained_name: str = Name of the pretrained model preferred
            max_length: int = Maximum length for the tokenizer
        Returns:
            Returns the tokenizer
        """
        self.tokenizer = AutoTokenizer.from_pretrained(pretrained_name)
        self.max_length = max_length
        # Add special tokens if not present (should be)
        self.pad_token_id = self.tokenizer.pad_token_id
        self.bos_token_id = self.tokenizer.bos_token_id
        self.mask_token_id = self.tokenizer.mask_token_id
        self.cls_token_id = self.tokenizer.cls_token_id
        self.sep_token_id = self.tokenizer.sep_token_id
        self.vocab_size = self.tokenizer.vocab_size

    def encode(self, text, max_length=None, padding=False, truncation=True):
        if max_length is None:
            max_length = self.max_length
        return self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=max_length,
            padding=padding,
            truncation=truncation,
            return_tensors=None,  # return lists instead of tensors
            return_attention_mask=True,
            return_token_type_ids=False,
        )

    def decode(self, ids, skip_special_tokens=True):
        return self.tokenizer.decode(ids, skip_special_tokens=skip_special_tokens)

    def get_vocab_size(self):
        return self.vocab_size


# Create an instance of the TokenizerWrapper
tokenizer_wrapper = TokenizerWrapper("bert-base-uncased")

# Define a sample text
sample_text = "This is an example sentence for tokenization."
print(f"Original Text: {sample_text}\n")

# Encode the text
encoded_output = tokenizer_wrapper.encode(sample_text)

# The encoded_output is a dictionary, extract input_ids and attention_mask
input_ids = encoded_output['input_ids']
attention_mask = encoded_output['attention_mask']

print(f"Encoded Input IDs: {input_ids}")
print(f"Attention Mask: {attention_mask}\n")

# Decode the input IDs back to text
decoded_text = tokenizer_wrapper.decode(input_ids)
print(f"Decoded Text: {decoded_text}")

# Also print vocabulary size
print(f"\nTokenizer Vocabulary Size: {tokenizer_wrapper.get_vocab_size()}")

In [ ]:
# Instantiate and test
tokenizer = TokenizerWrapper()
print(f"Vocabulary size: {tokenizer.get_vocab_size():,}")
print(f"Pad token ID: {tokenizer.pad_token_id}, Mask token ID: {tokenizer.mask_token_id}")
print(f"CLS token ID: {tokenizer.cls_token_id}")

sample_text = ("The Tower of London is a historic castle. It is a nice place indeed","I am new born in Christ")
encoded = tokenizer.encode(sample_text, max_length=32, padding=True)
print("\nEncoded sample:")
print(f"  input_ids: {encoded['input_ids']}")
print(f"  attention_mask: {encoded['attention_mask']}")
print("Decoded back:", tokenizer.decode(encoded['input_ids']))


### Dataset Creation


In [ ]:
import random
from typing import List, Tuple, Dict
import torch
from torch.utils.data import Dataset
from tqdm import tqdm

class WikiTextLayoutDataset(Dataset):
    """
    Converts a list of WikiText‑103 articles into training samples.
    Each sample: token IDs, normalised bounding boxes, attention mask, MLM labels.
    """

    def __init__(self, articles: List[str], tokenizer_wrapper, max_length: int = 512,
                 mlm_prob: float = 0.15, stride: int = None, max_articles: int = None):
        self.tokenizer = tokenizer_wrapper
        self.max_length = max_length
        self.mlm_prob = mlm_prob
        self.stride = stride if stride is not None else max_length // 2 #the stride is the sliding window over the tokens in each line

        self.samples: List[Dict] = []
        # Optionally limit articles for fast prototyping
        articles = articles[:max_articles] if max_articles else articles

        print(f"Processing {len(articles)} articles...")
        for article in tqdm(articles, desc="Chunking articles"):
            if not article.strip():
                continue
            # Tokenize the whole article into a long sequence of subword IDs and boxes
            token_ids, boxes = self._article_to_tokens_and_boxes(article)
            if len(token_ids) == 0:
                continue
            # Split into overlapping chunks with special tokens
            chunks = self._chunk_and_mask(token_ids, boxes)
            self.samples.extend(chunks)

    def _article_to_tokens_and_boxes(self, article: str) -> Tuple[List[int], List[Tuple[float,float,float,float]]]:
        """Tokenize article line by line, generating boxes for each subword token."""
        lines = article.split('\n')
        lines = [line.strip() for line in lines if line.strip()]
        if not lines:
            return [], []
        num_lines = len(lines)

        # First, compute subword token counts for all lines (with truncation)
        line_token_ids = []
        line_token_counts = []
        for line in lines:
            # Truncate to max_length to avoid long sequences
            ids = self.tokenizer.tokenizer.encode(
                line, add_special_tokens=False,
                truncation=True, max_length=self.max_length
            )
            line_token_ids.append(ids)
            line_token_counts.append(len(ids))

        max_tokens_line = max(line_token_counts) if line_token_counts else 1


        # Generate flat list of token IDs and corresponding boxes
        all_token_ids = []
        all_boxes = []
        #use the same technique to create the generic bboxes
        for line_idx, (line, ids) in enumerate(zip(lines, line_token_ids)):
            for token_idx, token_id in enumerate(ids):
                # Normalise 2D position
                x0 = token_idx / max_tokens_line
                x1 = (token_idx + 1) / max_tokens_line
                y0 = line_idx / num_lines
                y1 = (line_idx + 1) / num_lines
                all_token_ids.append(token_id)
                all_boxes.append((x0, y0, x1, y1))
        return all_token_ids, all_boxes

    def _chunk_and_mask(self, token_ids: List[int], boxes: List[Tuple]) -> List[Dict]:
        """Split long sequence into chunks of max_length, add special tokens, apply MLM."""
        chunks = []
        content_length = self.max_length - 2  # space for [CLS] and [SEP] special bert tokens
        start = 0
        while start < len(token_ids):
            end = min(start + content_length, len(token_ids))
            chunk_ids = token_ids[start:end]
            chunk_boxes = boxes[start:end]

            # Add special tokens
            input_ids = [self.tokenizer.cls_token_id] + chunk_ids + [self.tokenizer.sep_token_id]
            # Box for [CLS] and [SEP]: full page (0,0,1,1)
            cls_box = (0.0, 0.0, 1.0, 1.0)
            sep_box = (0.0, 0.0, 1.0, 1.0)
            bboxes = [cls_box] + chunk_boxes + [sep_box]
            attention_mask = [1] * len(input_ids)

            # Pad to max_length if needed
            pad_len = self.max_length - len(input_ids)
            if pad_len > 0:
                input_ids += [self.tokenizer.pad_token_id] * pad_len
                bboxes += [(0.0, 0.0, 0.0, 0.0)] * pad_len
                attention_mask += [0] * pad_len

            # Create MLM labels
            labels = self._create_mlm_labels(input_ids)

            chunks.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "bboxes": torch.tensor(bboxes, dtype=torch.float),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })

            # Slide window
            start += self.stride
            if start >= len(token_ids):
                break
        return chunks

    def _create_mlm_labels(self, input_ids: List[int]) -> List[int]:
        """
        Create labels for Masked Language Modelling.
        Replaces 15% of the non‑special tokens with [MASK] and returns labels
        (original tokens at masked positions, -100 elsewhere).
        """
        labels = [-100] * len(input_ids)
        candidate_indices = [
            i for i, tid in enumerate(input_ids)
            if tid not in (self.tokenizer.cls_token_id, self.tokenizer.sep_token_id, self.tokenizer.pad_token_id)
        ]
        random.shuffle(candidate_indices)
        num_masks = max(1, int(len(candidate_indices) * self.mlm_prob))
        mask_indices = candidate_indices[:num_masks]

        for idx in mask_indices:
            labels[idx] = input_ids[idx]
            r = random.random()
            if r < 0.8:
                input_ids[idx] = self.tokenizer.mask_token_id
            elif r < 0.9:
                input_ids[idx] = random.randrange(self.tokenizer.vocab_size)
            # else keep unchanged
        return labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

_article_to_tokens_and_boxes: Splits article into lines, tokenises each line with BERT subword tokenizer, and for each subword computes a normalised bounding box. The x‑coordinate is the token’s position within its line divided by the maximum number of tokens in any line of that article, y‑coordinate is the line index divided by total lines. This yields the grid layout we visualised earlier.

_chunk_and_mask: Slides a window of length max_length - 2 across the long token list, adds [CLS] at the start and [SEP] at the end (both with a “full page” box), pads if needed, and applies MLM masking via _create_mlm_labels.

_create_mlm_labels: Selects 15% of the non‑special tokens, stores the original token in labels (with -100 for others), and applies the 80‑10‑10 replacement strategy (mask/random/unchanged).

The dataset returns a dictionary with input_ids, bboxes, attention_mask, and labels as tensors.

In [ ]:
import torch

# Use a small subset of articles for demonstration to avoid long processing
sample_articles = articles[:50] # Use first 50 articles

print(f"Creating WikiTextLayoutDataset with {len(sample_articles)} articles...")
wiki_dataset = WikiTextLayoutDataset(
    articles=sample_articles,
    tokenizer_wrapper=tokenizer_wrapper,
    max_length=128, # Smaller max_length for a concise example
    mlm_prob=0.15,
    stride=64 # Overlapping chunks
)

print(f"Dataset created with {len(wiki_dataset)} samples.")

if len(wiki_dataset) > 0:
    # Get the first sample
    first_sample = wiki_dataset[0]

    print("\n--- First Sample from WikiTextLayoutDataset ---")
    print(f"Input IDs shape: {first_sample['input_ids'].shape}")
    print(f"Bounding Boxes shape: {first_sample['bboxes'].shape}")
    print(f"Attention Mask shape: {first_sample['attention_mask'].shape}")
    print(f"Labels shape: {first_sample['labels'].shape}")

    # Decode input_ids
    decoded_input = tokenizer_wrapper.decode(first_sample['input_ids'].tolist())
    print(f"\nDecoded Input (masked tokens): {decoded_input}")

    # Show some bounding boxes
    print("\nFirst 5 Bounding Boxes:")
    print(first_sample['bboxes'][:5].tolist())

    # Show attention mask
    print("\nFirst 10 Attention Mask values:")
    print(first_sample['attention_mask'][:10].tolist())

    # Find and decode original masked tokens from labels
    mlm_labels = first_sample['labels'].tolist()
    masked_tokens_info = []
    for i, label_id in enumerate(mlm_labels):
        if label_id != -100:
            original_token = tokenizer_wrapper.decode([label_id])
            current_input_token = tokenizer_wrapper.decode([first_sample['input_ids'].tolist()[i]])
            masked_tokens_info.append(f"Index {i}: Masked: '{current_input_token}' (originally '{original_token}')")
    if masked_tokens_info:
        print("\nMLM Labels (original tokens at masked positions):")
        for info in masked_tokens_info:
            print(info)
    else:
        print("\nNo tokens were masked in this sample (unlikely for mlm_prob > 0).")

else:
    print("Dataset is empty, cannot retrieve a sample.")

### Bounding Box Coordinates Distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Collect all bounding boxes from the dataset
all_bboxes = []
for i in range(len(wiki_dataset)):
    sample = wiki_dataset[i]
    # Filter out padding boxes (0.0, 0.0, 0.0, 0.0)
    valid_bboxes = sample['bboxes'][~torch.all(sample['bboxes'] == 0.0, dim=1)]
    all_bboxes.extend(valid_bboxes.tolist())

if not all_bboxes:
    print("No valid bounding boxes found in the dataset.")
else:
    all_bboxes = np.array(all_bboxes)

    # Extract x0, y0, x1, y1 coordinates
    x0_coords = all_bboxes[:, 0]
    y0_coords = all_bboxes[:, 1]
    x1_coords = all_bboxes[:, 2]
    y1_coords = all_bboxes[:, 3]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Distribution of Bounding Box Coordinates', fontsize=16)

    # Plot x0 distribution
    axes[0, 0].hist(x0_coords, bins=50, edgecolor='black', alpha=0.7)
    axes[0, 0].set_title('Distribution of X0 (Left)')
    axes[0, 0].set_xlabel('Normalized X-coordinate')
    axes[0, 0].set_ylabel('Frequency')

    # Plot y0 distribution
    axes[0, 1].hist(y0_coords, bins=50, edgecolor='black', alpha=0.7)
    axes[0, 1].set_title('Distribution of Y0 (Top)')
    axes[0, 1].set_xlabel('Normalized Y-coordinate')
    axes[0, 1].set_ylabel('Frequency')

    # Plot x1 distribution
    axes[1, 0].hist(x1_coords, bins=50, edgecolor='black', alpha=0.7)
    axes[1, 0].set_title('Distribution of X1 (Right)')
    axes[1, 0].set_xlabel('Normalized X-coordinate')
    axes[1, 0].set_ylabel('Frequency')

    # Plot y1 distribution
    axes[1, 1].hist(y1_coords, bins=50, edgecolor='black', alpha=0.7)
    axes[1, 1].set_title('Distribution of Y1 (Bottom)')
    axes[1, 1].set_xlabel('Normalized Y-coordinate')
    axes[1, 1].set_ylabel('Frequency')

    plt.tight_layout(rect=[0, 0.03, 1, 0.96]) # Adjust layout to prevent title overlap
    plt.show()

### Multihead Attention

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    Multi‑Head Scaled Dot‑Product Attention.
    """
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1, bias: bool = True):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.scale = math.sqrt(self.d_k)

        # Linear projections for Q, K, V and output
        self.W_q = nn.Linear(d_model, d_model, bias=bias)
        self.W_k = nn.Linear(d_model, d_model, bias=bias)
        self.W_v = nn.Linear(d_model, d_model, bias=bias)
        self.W_o = nn.Linear(d_model, d_model, bias=bias)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        """
        Args:
            query: (B, L_q, D)
            key:   (B, L_k, D)
            value: (B, L_k, D)
            mask:  optional attention mask (B, 1, 1, L_k) or (B, 1, L_q, L_k)
        Returns:
            output: (B, L_q, D)
            attention_weights: (B, num_heads, L_q, L_k) – for inspection
        """
        B, L_q, _ = query.shape
        _, L_k, _ = key.shape

        # Linear projections
        Q = self.W_q(query)  # (B, L_q, D)
        K = self.W_k(key)    # (B, L_k, D)
        V = self.W_v(value)  # (B, L_k, D)

        # Reshape to (B, num_heads, L, d_k)
        Q = Q.view(B, L_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled dot‑product attention
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (B, h, L_q, L_k)

        if mask is not None:
            # mask should be broadcastable to (B, h, L_q, L_k)
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)  # (B, h, L_q, L_k)
        attn_weights = self.dropout(attn_weights)

        # Weighted sum
        attn_output = torch.matmul(attn_weights, V)  # (B, h, L_q, d_k)

        # Concatenate heads and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, L_q, self.d_model)
        output = self.W_o(attn_output)

        return output, attn_weights

### Position-Wise Feedforward

In [ ]:
# Cell 9: Position-wise Feed-Forward Network
class FeedForward(nn.Module):
    """
    Standard two‑layer MLP with GELU activation, applied to each position separately.
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Args:
            x: (B, L, D)
        Returns:
            (B, L, D)
        """
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))

### Transformer Encoder Block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """
    A single encoder block: Self-Attention + Feed-Forward with residual connections and layer norm.
    Uses post-norm (LayerNorm after the residual addition) as in the original Transformer / BERT.
    """
    def __init__(self, d_model: int, num_heads: int, d_ff: int,
                 dropout: float = 0.1, bias: bool = True):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout, bias)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x: (B, L, D)
            mask: optional attention mask (B, 1, 1, L) or (B, 1, L, L)
        Returns:
            (B, L, D)
        """
        # Self-attention
        attn_out, _ = self.self_attn(x, x, x, mask) # Unpack only the output, discard weights
        x = x + self.dropout(attn_out)
        x = self.norm1(x)

        # Feed-forward
        ff_out = self.ff(x)
        x = x + self.dropout(ff_out)
        x = self.norm2(x)
        return x

### Spatial Embedding Layer

In [ ]:
class SpatialEmbedding(nn.Module):
    """
    Converts normalised bounding box coordinates (x0, y0, x1, y1) into a dense embedding
    of size d_model. This embedding is added to the token embedding to provide layout information.
    """
    def __init__(self, d_model: int, hidden_dim: int = None, dropout: float = 0.1):
        super().__init__()
        hidden_dim = hidden_dim or d_model
        self.mlp = nn.Sequential(
            nn.Linear(4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model)
        )

    def forward(self, bboxes: torch.Tensor) -> torch.Tensor:
        """
        Args:
            bboxes: (batch_size, seq_len, 4) – normalised coordinates [x0,y0,x1,y1] in [0,1]
        Returns:
            (batch_size, seq_len, d_model)
        """
        return self.mlp(bboxes)

### Layout-aware transfomer block

In [ ]:
class LayoutTransformerEncoder(nn.Module):
    """
    Transformer encoder that fuses text tokens with spatial layout (bounding boxes).
    Combines:
      - Learned token embeddings
      - Spatial embedding from bounding boxes
      - Learned positional embeddings (up to max_len)
      - Stack of TransformerEncoderBlock layers
    """
    def __init__(self, vocab_size: int, d_model: int = 768, num_heads: int = 12,
                 num_layers: int = 12, d_ff: int = 3072, max_len: int = 512,
                 dropout: float = 0.1, bias: bool = True):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len

        # Token embedding (includes padding idx = 0)
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        # Spatial embedding (bounding boxes -> d_model)
        self.spatial_embedding = SpatialEmbedding(d_model, dropout=dropout)

        # Learned positional embedding (for absolute position in sequence)
        self.position_embedding = nn.Embedding(max_len, d_model)

        # Stack of encoder blocks
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout, bias)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)  # final norm

    def forward(self, input_ids, bboxes, attention_mask=None):
        """
        Args:
            input_ids: (B, L) – token IDs
            bboxes: (B, L, 4) – normalised bounding boxes
            attention_mask: (B, L) – 1 for real tokens, 0 for padding (optional)
        Returns:
            hidden_states: (B, L, d_model)
        """
        B, L = input_ids.shape
        device = input_ids.device

        # 1. Token embeddings
        tok_emb = self.token_embedding(input_ids) * math.sqrt(self.d_model)  # scale

        # 2. Spatial embeddings
        spa_emb = self.spatial_embedding(bboxes)

        # 3. Positional embeddings (absolute positions)
        positions = torch.arange(L, dtype=torch.long, device=device).unsqueeze(0)  # (1, L)
        pos_emb = self.position_embedding(positions)

        # Sum all embeddings
        x = tok_emb + spa_emb + pos_emb
        x = self.dropout(x)

        # 4. Create attention mask (shape for MultiHeadAttention)
        if attention_mask is not None:
            # Convert (B, L) -> (B, 1, 1, L) with 0/1 -> 0 = mask out
            extended_mask = attention_mask[:, None, None, :]  # (B, 1, 1, L)
        else:
            extended_mask = None

        # 5. Pass through encoder blocks
        for layer in self.layers:
            x = layer(x, mask=extended_mask)

        # Final layer norm
        x = self.layer_norm(x)
        return x

### MLM Head

In [ ]:
class MLMHead(nn.Module):
    """
    Linear layer that maps encoder hidden states to logits over the vocabulary.
    Used for pre‑training with Masked Language Modelling.
    """
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.linear = nn.Linear(d_model, vocab_size)

    def forward(self, hidden_states):
        """
        Args:
            hidden_states: (B, L, d_model)
        Returns:
            logits: (B, L, vocab_size)
        """
        return self.linear(hidden_states)

### NER Head for token classification

In [ ]:
# Cell 14: NER Head for token classification (e.g., BIO tagging)
class NERHead(nn.Module):
    """
    Linear classifier on top of encoder outputs for token‑level entity extraction.
    """
    def __init__(self, d_model: int, num_labels: int):
        super().__init__()
        self.classifier = nn.Linear(d_model, num_labels)

    def forward(self, hidden_states):
        """
        Args:
            hidden_states: (B, L, d_model)
        Returns:
            logits: (B, L, num_labels)
        """
        return self.classifier(hidden_states)

### Full RIFD Model

In [ ]:
# Cell 15: RIFD – Read Info From Documents (complete model)
class RIFD(nn.Module):
    """
    Layout‑aware Transformer model that can be used for:
    - Pre‑training: Masked Language Modelling (MLM)
    - Fine‑tuning: Token classification (NER on forms)

    Combines the LayoutTransformerEncoder with task‑specific heads.
    """
    def __init__(self, vocab_size: int, d_model: int = 768, num_heads: int = 12,
                 num_layers: int = 12, d_ff: int = 3072, max_len: int = 512,
                 dropout: float = 0.1, bias: bool = True, num_ner_labels: int = None):
        super().__init__()
        self.encoder = LayoutTransformerEncoder(
            vocab_size=vocab_size,
            d_model=d_model,
            num_heads=num_heads,
            num_layers=num_layers,
            d_ff=d_ff,
            max_len=max_len,
            dropout=dropout,
            bias=bias
        )
        self.mlm_head = MLMHead(d_model, vocab_size)

        # NER head is created only when num_ner_labels is provided (e.g., during fine‑tuning)
        if num_ner_labels is not None:
            self.ner_head = NERHead(d_model, num_ner_labels)
        else:
            self.ner_head = None

    def forward_mlm(self, input_ids, bboxes, attention_mask=None):
        """
        Returns logits for MLM: (B, L, vocab_size)
        """
        hidden = self.encoder(input_ids, bboxes, attention_mask)
        return self.mlm_head(hidden)

    def forward_ner(self, input_ids, bboxes, attention_mask=None):
        """
        Returns logits for token classification: (B, L, num_labels)
        """
        if self.ner_head is None:
            raise ProjectException("NER head has not been initialised (num_ner_labels was None).")
        hidden = self.encoder(input_ids, bboxes, attention_mask)
        return self.ner_head(hidden)

    def forward(self, input_ids, bboxes, attention_mask=None, task="mlm"):
        """
        Unified forward for convenience.
        """
        # Temporary fix for torchsummary: cast float input_ids to long
        if input_ids.dtype == torch.float:
            input_ids = input_ids.long()

        if task == "mlm":
            return self.forward_mlm(input_ids, bboxes, attention_mask)
        elif task == "ner":
            return self.forward_ner(input_ids, bboxes, attention_mask)
        else:
            raise ProjectException(f"Unknown task: {task}")

### FUNSD Dataset for NER fine‑tuning

In [ ]:
import torch
from torch.utils.data import Dataset
from pathlib import Path
import json
import cv2
from typing import List, Dict, Tuple, Optional

class FUNSDDataset(Dataset):
    """
    Loads FUNSD documents, tokenises words, aligns BIO labels to subword tokens,
    and returns samples ready for the RIFD model.

    Label scheme (BIO):
        B-QUESTION, I-QUESTION, B-ANSWER, I-ANSWER,
        B-HEADER,   I-HEADER,   B-OTHER,    I-OTHER
        plus ignore index -100 for special tokens and padding.
    """
    # Map from FUNSD entity type to BIO prefix indices
    ENTITY_TO_BIO = {
        "question": 0,   # 0/1 -> B-QUESTION, I-QUESTION
        "answer":   2,   # 2/3 -> B-ANSWER,   I-ANSWER
        "header":   4,   # 4/5 -> B-HEADER,   I-HEADER
        "other":    6    # 6/7 -> B-OTHER,    I-OTHER
    }
    IGNORE_INDEX = -100

    def __init__(self, data_dir: Path, tokenizer_wrapper, max_length: int = 512,
                 split: str = "train"):
        self.data_dir = Path(data_dir)
        self.tokenizer = tokenizer_wrapper
        self.max_length = max_length
        self.split = split   # "train" or "test"

        # Collect (image_path, annotation_path) pairs
        self.samples: List[Tuple[Path, Path]] = []
        self._find_samples()

    def _find_samples(self):
        """Locate all image+annotation pairs, filtering by split."""
        # Look for training_data and testing_data subdirectories
        for subdir_name, target_split in [("training_data", "train"), ("testing_data", "test")]:
            if target_split != self.split:
                continue
            subdir = self.data_dir / subdir_name
            if not subdir.exists():
                # Try dataset/training_data etc.
                subdir = self.data_dir / "dataset" / subdir_name
            if not subdir.exists():
                continue
            # Images are usually in "images", annotations in "annotations"
            img_dir = subdir / "images"
            ann_dir = subdir / "annotations"
            if not img_dir.exists() or not ann_dir.exists():
                # Some versions have flat layout: all files in subdir
                img_dir = subdir
                ann_dir = subdir
            # Find all images
            for img_path in sorted(img_dir.glob("*.png")):
                ann_path = ann_dir / f"{img_path.stem}.json"
                if not ann_path.exists():
                    # try in another location
                    found = list(subdir.rglob(f"{img_path.stem}.json"))
                    if found:
                        ann_path = found[0]
                    else:
                        continue
                self.samples.append((img_path, ann_path))
        # If no subdir structure, just use all images under data_dir
        if not self.samples:
            all_imgs = sorted(self.data_dir.rglob("*.png"))
            for img_path in all_imgs:
                ann_path = img_path.with_suffix(".json")
                if not ann_path.exists():
                    found = list(self.data_dir.rglob(f"{img_path.stem}.json"))
                    if found:
                        ann_path = found[0]
                    else:
                        continue
                self.samples.append((img_path, ann_path))
        print(f"[FUNSDDataset] Found {len(self.samples)} samples for split '{self.split}'")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, ann_path = self.samples[idx]
        # Load image to get dimensions (for box normalisation)
        img = cv2.imread(str(img_path))
        if img is None:
            raise ProjectException(f"Failed to load image {img_path}")
        h, w = img.shape[:2]

        # Load annotation
        with open(ann_path, "r", encoding="utf-8") as f:
            annot = json.load(f)

        # Build list of words with text, normalised box, BIO label
        words = []
        for form_item in annot.get("form", []):
            entity_label = form_item.get("label", "other")
            word_list = form_item.get("words", [])
            for i, word_info in enumerate(word_list):
                box = word_info["box"]  # pixel coords [x0, y0, x1, y1]
                # Normalise to [0,1]
                x0, y0, x1, y1 = box
                norm_box = [x0 / w, y0 / h, x1 / w, y1 / h]
                # BIO: first word in entity gets B-, others I-
                bio_prefix = "B" if i == 0 else "I"
                label_str = f"{bio_prefix}-{entity_label.upper()}"
                words.append({
                    "text": word_info["text"],
                    "box": norm_box,
                    "label": label_str
                })

        return self._tokenize_and_align(words, img_path.name)

    def _tokenize_and_align(self, words: List[Dict], doc_name: str) -> Dict:
        """Tokenize all words, align labels with subwords, add special tokens."""
        # Pre-allocate lists
        input_ids = [self.tokenizer.cls_token_id]
        bboxes = [[0.0, 0.0, 1.0, 1.0]]  # [CLS] box
        labels = [self.IGNORE_INDEX]

        for w in words:
            # Tokenize single word (no special tokens)
            sub_ids = self.tokenizer.tokenizer.encode(w["text"], add_special_tokens=False)
            if not sub_ids:
                continue
            # Determine label index for each subword
            # The word's label string like "B-QUESTION" -> we need an integer
            bio_label = w["label"]
            # Map to integer
            label_int = self._bio_label_to_int(bio_label)

            # First subword gets the original label (which may be B-... or I-...)
            # Subsequent subwords always get I- variant of the same entity
            for i, tid in enumerate(sub_ids):
                input_ids.append(tid)
                # Bounding box: same as the word's box
                bboxes.append(w["box"])
                if i == 0:
                    labels.append(label_int)
                else:
                    # Convert to I- label if original was B-, else keep I-
                    # label_int % 2 == 0 means B-; we set to I- (B_idx+1)
                    if label_int % 2 == 0:  # B- label
                        labels.append(label_int + 1)  # I- variant
                    else:
                        labels.append(label_int)  # already I-

        # Add [SEP]
        input_ids.append(self.tokenizer.sep_token_id)
        bboxes.append([0.0, 0.0, 1.0, 1.0])
        labels.append(self.IGNORE_INDEX)

        # Store the length of the actual content before padding/truncation
        effective_seq_len_for_mask = len(input_ids)

        # Truncate or pad to max_length
        if len(input_ids) > self.max_length:
            input_ids = input_ids[:self.max_length]
            bboxes = bboxes[:self.max_length]
            labels = labels[:self.max_length]
            effective_seq_len_for_mask = self.max_length # Update effective length if truncated
        else:
            pad_len = self.max_length - len(input_ids)
            input_ids += [self.tokenizer.pad_token_id] * pad_len
            bboxes += [[0.0, 0.0, 0.0, 0.0]] * pad_len
            labels += [self.IGNORE_INDEX] * pad_len

        # Attention mask: 1 for real tokens, 0 for padding
        # Use the corrected effective_seq_len_for_mask
        attention_mask = [1] * effective_seq_len_for_mask + [0] * (self.max_length - effective_seq_len_for_mask)
        # --- BUG FIX END ---

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "bboxes": torch.tensor(bboxes, dtype=torch.float),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

    def _bio_label_to_int(self, bio_str: str) -> int:
        """Convert BIO label string to integer index."""
        # e.g., "B-QUESTION" -> 0, "I-QUESTION" -> 1, "B-ANSWER" -> 2, etc.
        bio, entity = bio_str.split("-")
        entity = entity.lower()
        base = self.ENTITY_TO_BIO[entity]
        return base if bio == "B" else base + 1

    @classmethod
    def get_num_labels(cls) -> int:
        """Total number of BIO labels (including the two per entity)."""
        return 8  # B/I for four entities

    @classmethod
    def get_label_list(cls) -> List[str]:
        """Return the list of label names."""
        entities = ["QUESTION", "ANSWER", "HEADER", "OTHER"]
        return [f"{b}-{e}" for e in entities for b in ["B", "I"]]


In [ ]:
import torch

# Instantiate FUNSDDataset for the 'train' split
funsd_train_dataset = FUNSDDataset(
    data_dir=DATA_DIR / "funsd",
    tokenizer_wrapper=tokenizer_wrapper,
    max_length=128, # Use a smaller max_length for easier inspection
    split="train"
)

print(f"FUNSD training dataset created with {len(funsd_train_dataset)} samples.")

if len(funsd_train_dataset) > 0:
    # Get the first sample from the FUNSD dataset
    first_funsd_sample = funsd_train_dataset[0]

    print("\n--- First Sample from FUNSDDataset ---")
    print(f"Input IDs shape: {first_funsd_sample['input_ids'].shape}")
    print(f"Bounding Boxes shape: {first_funsd_sample['bboxes'].shape}")
    print(f"Attention Mask shape: {first_funsd_sample['attention_mask'].shape}")
    print(f"Labels shape: {first_funsd_sample['labels'].shape}")

    # Decode input_ids
    decoded_input_funsd = tokenizer_wrapper.decode(first_funsd_sample['input_ids'].tolist())
    print(f"\nDecoded Input: {decoded_input_funsd}")

    # Show some bounding boxes
    print("\nFirst 10 Bounding Boxes:")
    for i, bbox in enumerate(first_funsd_sample['bboxes'][:10].tolist()):
        print(f"  Token {i}: {bbox}")

    # Show attention mask
    print("\nFirst 10 Attention Mask values:")
    print(first_funsd_sample['attention_mask'][:10].tolist())

    # Decode and print labels
    label_map = {idx: label_name for idx, label_name in enumerate(FUNSDDataset.get_label_list())}
    label_map[FUNSDDataset.IGNORE_INDEX] = "[IGNORE]"

    print("\nFirst 10 Labels (decoded):")
    labels_decoded = [label_map[l.item()] for l in first_funsd_sample['labels'][:10]]
    print(labels_decoded)

    print("\nFull Decoded Text with Labels (first 20 tokens):")
    input_ids_list = first_funsd_sample['input_ids'].tolist()
    labels_list = first_funsd_sample['labels'].tolist()
    for i in range(min(20, len(input_ids_list))):
        token_text = tokenizer_wrapper.decode([input_ids_list[i]], skip_special_tokens=False)
        label_text = label_map[labels_list[i]]
        print(f"  {token_text:<15} : {label_text}")

else:
    print("FUNSD Dataset is empty, cannot retrieve a sample.")

In [ ]:
import torch.nn as nn

class MLMLoss(nn.Module):
    """
    Masked Language Modelling loss: cross‑entropy on masked positions only.
    Expects labels with -100 for positions to ignore.
    """
    def __init__(self, ignore_index: int = -100):
        super().__init__()
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def forward(self, logits, labels):
        """
        Args:
            logits: (B, L, vocab_size)
            labels: (B, L) with -100 for non‑masked tokens
        Returns:
            scalar loss
        """
        # Flatten to (B*L, vocab_size) and (B*L)
        return self.loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))


class TokenClassificationLoss(nn.Module):
    """
    Token‑level classification loss (e.g., for NER BIO tagging).
    Ignores positions with label -100.
    """
    def __init__(self, ignore_index: int = -100):
        super().__init__()
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def forward(self, logits, labels):
        """
        Args:
            logits: (B, L, num_labels)
            labels: (B, L) with -100 for special/padding tokens
        Returns:
            scalar loss
        """
        return self.loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))

In [ ]:
!pip install torchinfo

In [ ]:
# Configuration for instatiating the model
d_model = 256
num_heads = 16
num_layers = 16
d_ff = 1024
max_len = 512
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Instantiate the RIFD model (with NER head for full overview)
# tokenizer is available from previous cells
vocab_size = tokenizer.get_vocab_size()
# FUNSDDataset is available
num_ner_labels = FUNSDDataset.get_num_labels()

print(f"Instantiating RIFD model for summary with vocab_size={vocab_size}, num_ner_labels={num_ner_labels}...")
model_for_summary = RIFD(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    max_len=max_len,
    dropout=0.1,
    num_ner_labels=num_ner_labels
).to(device)


from torchinfo import summary

summary(
    model_for_summary,
    input_data=[torch.zeros(1, 512, dtype=torch.long),#input_ids  (batch, seq_len)
                torch.zeros(1, 512, 4, dtype=torch.float), #bboxes (batch, seq_len, bbox coordinates)
                torch.zeros(1, 512, dtype=torch.long)], #attention_mask (batch,seq_len)
    dtypes=[torch.long, torch.float, torch.long],
    device=str(device)
)

In [ ]:
#get model parameter size
from prettytable import PrettyTable

import torch



def model_memory_analysis(model):
    """Print per‑module parameter count and estimate training memory."""
    table = PrettyTable(["Module", "Parameters", "Size (KB/MB)"])
    total_params = 0
    total_bytes = 0

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        num = param.numel()
        bytes_ = num * param.element_size()  # 4 for float32
        # Choose unit
        if bytes_ < 1024:
            size_str = f"{bytes_} B"
        elif bytes_ < 1024**2:
            size_str = f"{bytes_/1024:.2f} KB"
        else:
            size_str = f"{bytes_/(1024**2):.4f} MB"
        table.add_row([name, num, size_str])
        total_params += num
        total_bytes += bytes_

    total_mb = total_bytes / (1024 ** 2)
    print(f"Total trainable parameters: {total_params:,}")
    print(f"Parameter memory (fp32): {total_mb:.2f} MB")
    # Training memory: parameters + gradients + Adam moments (2) = 4× parameter memory
    print(f"Estimated training memory (params + grads + Adam): {total_mb*4:.2f} MB")
    print("(Activations not included – they scale with batch size and sequence length.)")
    return table

print("=" * 80)
print("Model parameter breakdown")
print("=" * 80)
table = model_memory_analysis(model_for_summary)
print(table)

In [ ]:
print("Re-instantiating FUNSD training dataset to apply the fix...")
funsd_train_dataset = FUNSDDataset(
    data_dir=DATA_DIR / "funsd",
    tokenizer_wrapper=tokenizer_wrapper,
    max_length=128,
    split="train"
)

print(f"FUNSD training dataset re-created with {len(funsd_train_dataset)} samples.")

# Get a sample for model input
if len(funsd_train_dataset) > 0:
    sample_input = funsd_train_dataset[0]

    # Prepare input tensors with a batch dimension
    input_ids = sample_input['input_ids'].unsqueeze(0)
    bboxes = sample_input['bboxes'].unsqueeze(0)
    attention_mask = sample_input['attention_mask'].unsqueeze(0)

    print("\n--- Sample Input Tensor Shapes ---")
    print(f"input_ids shape: {input_ids.shape}")
    print(f"bboxes shape: {bboxes.shape}")
    print(f"attention_mask shape: {attention_mask.shape}")
    print(f"labels shape: {sample_input['labels'].shape}") # Labels are not input to model

    # Instantiate the RIFD model for NER
    vocab_size = tokenizer_wrapper.get_vocab_size()
    num_ner_labels = FUNSDDataset.get_num_labels()

    print(f"\nInstantiating RIFD model with vocab_size={vocab_size}, num_ner_labels={num_ner_labels}...")
    rifd_model = RIFD(
        vocab_size=vocab_size,
        num_ner_labels=num_ner_labels # Important for NER head
    )

    print("\n--- RIFD Model Architecture ---")
    print(rifd_model)

    # Perform a forward pass for NER task
    print("\nPerforming forward pass for NER task...")
    with torch.no_grad(): # No need to calculate gradients for just viewing output
        ner_logits = rifd_model.forward_ner(input_ids, bboxes, attention_mask)

    print("\n--- Model Output Shape (NER logits) ---")
    print(f"NER logits shape: {ner_logits.shape}")

else:
    print("FUNSD Dataset is empty after re-creation, cannot proceed with model demonstration.")


### Pretraining Loop

In [ ]:
import time
import math
import torch
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

class PreTrainer:
    """
    Handles MLM pre‑training of the RIFD model on a layout‑aware text corpus.

    Tracks loss in TensorBoard, saves best model (by validation loss) as best-dpt-model.pth.
    """
    def __init__(self, model, train_dataset, val_dataset,
                 mlm_loss_fn, tokenizer_wrapper,
                 batch_size=8, lr=5e-5, weight_decay=0.01,
                 grad_clip=1.0, num_epochs=10, device='cuda',
                 log_dir='runs/pretrain', save_path='best-dpt-model.pth'):
        self.model = model.to(device)
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.loss_fn = mlm_loss_fn
        self.tokenizer = tokenizer_wrapper
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.grad_clip = grad_clip
        self.num_epochs = num_epochs
        self.device = device
        self.save_path = save_path

        self.writer = SummaryWriter(log_dir=log_dir)

        # Dataloaders
        self.train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        self.val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Optimizer & scheduler
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=num_epochs)

        self.best_val_loss = float('inf')

    def train(self):
        global_step = 0
        for epoch in range(1, self.num_epochs + 1):
            self.model.train()
            train_loss = 0.0
            train_steps = 0
            epoch_start = time.time()

            pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}/{self.num_epochs} [Train]")
            for batch in pbar:
                input_ids = batch['input_ids'].to(self.device)
                bboxes = batch['bboxes'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                self.optimizer.zero_grad()
                logits = self.model.forward_mlm(input_ids, bboxes, attention_mask)
                loss = self.loss_fn(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.optimizer.step()

                train_loss += loss.item()
                train_steps += 1
                global_step += 1

                pbar.set_postfix({'loss': train_loss / train_steps})

                # Log training loss every 50 steps
                if global_step % 50 == 0:
                    self.writer.add_scalar('Loss/train', loss.item(), global_step)
                    # Also log perplexity = exp(loss)
                    self.writer.add_scalar('Perplexity/train', math.exp(loss.item()), global_step)

            avg_train_loss = train_loss / train_steps
            epoch_time = time.time() - epoch_start

            # Validation
            val_loss, val_perplexity = self._validate(epoch, global_step)

            # Scheduler step
            self.scheduler.step()

            # Print summary
            print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Val PPL: {val_perplexity:.2f} | Time: {epoch_time:.1f}s")

            # Save best model
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save(self.model.state_dict(), self.save_path)
                print(f"  → Best model saved (val_loss={val_loss:.4f})")

        self.writer.close()
        print("Pre‑training complete.")

    def _validate(self, epoch, global_step):
        self.model.eval()
        total_loss = 0.0
        total_steps = 0
        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch['input_ids'].to(self.device)
                bboxes = batch['bboxes'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                logits = self.model.forward_mlm(input_ids, bboxes, attention_mask)
                loss = self.loss_fn(logits, labels)
                total_loss += loss.item()
                total_steps += 1

        avg_loss = total_loss / total_steps if total_steps > 0 else 0.0
        ppl = math.exp(avg_loss) if avg_loss < float('inf') else float('inf')

        # Log to TensorBoard
        self.writer.add_scalar('Loss/val', avg_loss, global_step)
        self.writer.add_scalar('Perplexity/val', ppl, global_step)
        return avg_loss, ppl

In [ ]:
!nvidia-smi

In [ ]:
import gc
import torch

def clear_gpu_memory(device='cuda'):
    """Free all GPU memory currently occupied by tensors and caches."""
    if torch.cuda.is_available():
        # Delete any tensors still referenced in this notebook (globals)
        for obj in list(globals().values()):
            if isinstance(obj, torch.Tensor) and obj.is_cuda:
                del obj
        # Run garbage collector
        gc.collect()
        # Empty PyTorch cache
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        print("GPU memory cleared and stats reset.")
    else:
        print("No GPU available.")

# call clear_gpu_memory() whenever you want a fresh start
clear_gpu_memory()

In [ ]:
import torch
from torch.utils.data import random_split


# 1. Configuration (adjust for your GPU memory)

d_model = 256         # smaller for faster training; use 768 for full
num_heads = 4
num_layers = 4
d_ff = 1024
max_len = 512
batch_size = 16
lr = 5e-5
num_epochs = 5
mlm_prob = 0.15
stride = 256           # sliding window stride for chunking
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# 2. Build the pre‑training dataset
full_dataset = WikiTextLayoutDataset(
    articles=articles,
    tokenizer_wrapper=tokenizer,
    max_length=max_len,
    mlm_prob=mlm_prob,
    stride=stride
)
print(f"Total pre‑training samples: {len(full_dataset)}")

# Train/validation split (90/10)
val_size = max(1, int(0.1 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")


# 3. Create the RIFD model (no NER head)
rifd_model = RIFD(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    max_len=max_len,
    dropout=0.1,
    num_ner_labels=None          # only MLM head
)
print(f"Model parameters: {sum(p.numel() for p in rifd_model.parameters()):,}")


# 4. Loss and trainer
mlm_loss = MLMLoss(ignore_index=-100)

trainer = PreTrainer(
    model=rifd_model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    mlm_loss_fn=mlm_loss,
    tokenizer_wrapper=tokenizer,
    batch_size=batch_size,
    lr=lr,
    weight_decay=0.01,
    grad_clip=1.0,
    num_epochs=num_epochs,
    device=device,
    log_dir='runs/pretrain',
    save_path='best-dpt-model.pth'
)


# 5. Start training
trainer.train()

In [ ]:
#tensorboard
%load_ext tensorboard
%tensorboard --logdir runs/pretrain

### Finetuner for the DPT

In [ ]:
!pip install seqeval

In [ ]:
clear_gpu_memory()

In [ ]:
# Cell 19: Fine‑tuning Trainer (Token Classification)
import numpy as np
from seqeval.metrics import classification_report, f1_score as seq_f1_score

class FineTuner:
    """
    Fine‑tunes the pre‑trained RIFD model on a token‑classification task (FUNSD).
    Uses the NER head, tracks metrics (loss, token accuracy, entity F1) in TensorBoard,
    and saves the best model as best-ner-model.pth.
    """
    def __init__(self, model, train_dataset, val_dataset,
                 tokenizer_wrapper, label_list,
                 batch_size=8, lr=5e-5, weight_decay=0.01,
                 grad_clip=1.0, num_epochs=10, device='cuda',
                 log_dir='runs/finetune', save_path='best-ner-model.pth'):
        self.device = device
        self.model = model.to(device)
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.tokenizer = tokenizer_wrapper
        self.label_list = label_list          # e.g., ["B-QUESTION", "I-QUESTION", ...]
        self.num_labels = len(label_list)
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.grad_clip = grad_clip
        self.num_epochs = num_epochs
        self.save_path = save_pathcode .

        # Ensure NER head exists
        if model.ner_head is None:
            raise ProjectException("Model does not have an NER head; set num_ner_labels when creating RIFD.")
        # Ensure label count matches
        assert model.ner_head.classifier.out_features == self.num_labels, \
            f"NER head expects {model.ner_head.classifier.out_features} labels, but dataset provides {self.num_labels}"

        self.loss_fn = TokenClassificationLoss(ignore_index=FUNSDDataset.IGNORE_INDEX)

        self.writer = SummaryWriter(log_dir=log_dir)

        # DataLoaders
        self.train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        self.val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Optimizer & scheduler
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=num_epochs)

        self.best_val_f1 = 0.0

    def train(self):
        global_step = 0
        for epoch in range(1, self.num_epochs + 1):
            self.model.train()
            train_loss = 0.0
            train_steps = 0

            pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}/{self.num_epochs} [Train]")
            for batch in pbar:
                input_ids = batch['input_ids'].to(self.device)
                bboxes = batch['bboxes'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                self.optimizer.zero_grad()
                logits = self.model.forward_ner(input_ids, bboxes, attention_mask)
                loss = self.loss_fn(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.optimizer.step()

                train_loss += loss.item()
                train_steps += 1
                global_step += 1

                pbar.set_postfix({'loss': train_loss / train_steps})

                if global_step % 50 == 0:
                    self.writer.add_scalar('Loss/train', loss.item(), global_step)

            avg_train_loss = train_loss / train_steps

            # Validation
            val_loss, token_acc, entity_f1 = self._validate(epoch, global_step)
            self.scheduler.step()

            print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Token Acc: {token_acc:.4f} | Entity F1: {entity_f1:.4f}")

            # Save best model by entity F1
            if entity_f1 > self.best_val_f1:
                self.best_val_f1 = entity_f1
                torch.save(self.model.state_dict(), self.save_path)
                print(f"  → Best model saved (entity F1={entity_f1:.4f})")

        self.writer.close()
        print("Fine‑tuning complete.")

    def _validate(self, epoch, global_step):
        self.model.eval()
        total_loss = 0.0
        total_steps = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch['input_ids'].to(self.device)
                bboxes = batch['bboxes'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                logits = self.model.forward_ner(input_ids, bboxes, attention_mask)
                loss = self.loss_fn(logits, labels)
                total_loss += loss.item()
                total_steps += 1

                # Predictions
                preds = torch.argmax(logits, dim=-1)  # (B, L)

                # Convert to lists of label strings for seqeval (masking ignored indices)
                for i in range(labels.size(0)):
                    true = labels[i].cpu().tolist()
                    pred = preds[i].cpu().tolist()
                    # Filter out special tokens and padding (ignore index = -100)
                    true_labels = []
                    pred_labels = []
                    for t, p in zip(true, pred):
                        if t != FUNSDDataset.IGNORE_INDEX:
                            true_labels.append(self.label_list[t])
                            pred_labels.append(self.label_list[p])
                    all_labels.append(true_labels)
                    all_preds.append(pred_labels)

        avg_loss = total_loss / total_steps if total_steps > 0 else 0.0

        # Token‑level accuracy (only on non‑ignored tokens)
        flat_true = [t for seq in all_labels for t in seq]
        flat_pred = [p for seq in all_preds for p in seq]
        token_acc = sum(1 for t, p in zip(flat_true, flat_pred) if t == p) / len(flat_true) if flat_true else 0.0

        # Entity‑level F1 (using seqeval)
        try:
            entity_f1 = seq_f1_score(all_labels, all_preds, average='micro')
        except Exception:
            entity_f1 = 0.0

        # Log
        self.writer.add_scalar('Loss/val', avg_loss, global_step)
        self.writer.add_scalar('TokenAcc/val', token_acc, global_step)
        self.writer.add_scalar('EntityF1/val', entity_f1, global_step)

        return avg_loss, token_acc, entity_f1

In [ ]:
import torch
from torch.utils.data import random_split


# 1. Configuration (must match pre‑trained model)
d_model = 256          # same as pre‑training
num_heads = 4
num_layers = 4
d_ff = 1024
max_len = 512
batch_size = 16          # FUNSD samples can be long, reduce if OOM
lr = 5e-5
num_epochs = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


# 2. Build the FUNSD datasets
train_dataset = FUNSDDataset(
    data_dir=DATA_DIR / "funsd",
    tokenizer_wrapper=tokenizer,
    max_length=max_len,
    split="train"
)
test_dataset = FUNSDDataset(
    data_dir=DATA_DIR / "funsd",
    tokenizer_wrapper=tokenizer,
    max_length=max_len,
    split="test"
)
print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")


# 3. Create model with NER head and load pre‑trained encoder
num_ner_labels = FUNSDDataset.get_num_labels()
label_list = FUNSDDataset.get_label_list()
print(f"NER labels: {label_list}")

# Instantiate with the same encoder architecture but with NER head
ner_model = RIFD(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    max_len=max_len,
    dropout=0.1,
    num_ner_labels=num_ner_labels
)

# Load pre‑trained encoder weights (ignore missing/unexpected keys for NER head)
if not Path('best-dpt-model.pth').exists():
    raise ProjectException("Pre‑trained checkpoint 'best-dpt-model.pth' not found. Run pre‑training first.")

pretrained_state = torch.load('best-dpt-model.pth', map_location=device)
# Filter out NER head keys if any (they shouldn't exist in pre‑trained checkpoint)
encoder_state = {k: v for k, v in pretrained_state.items() if not k.startswith('ner_head')}
missing, unexpected = ner_model.load_state_dict(encoder_state, strict=False)
print(f"Loaded encoder weights. Missing keys (expected): {len(missing)}, Unexpected keys: {len(unexpected)}")
# Missing keys should be only 'ner_head.*' – that's fine

ner_model.to(device)


# 4. Loss & Trainer
ner_loss = TokenClassificationLoss(ignore_index=FUNSDDataset.IGNORE_INDEX)

finetuner = FineTuner(
    model=ner_model,
    train_dataset=train_dataset,
    val_dataset=test_dataset,          # use test as validation (FUNSD has no public val split)
    tokenizer_wrapper=tokenizer,
    label_list=label_list,
    batch_size=batch_size,
    lr=lr,
    weight_decay=0.01,
    grad_clip=1.0,
    num_epochs=num_epochs,
    device=device,
    log_dir='runs/finetune',
    save_path='best-ner-model.pth'
)

# 5. Start fine‑tuning
finetuner.train()

### Inference

In [ ]:
# Cell 20: Inference function and demo
import torch
import cv2
import json
from typing import List, Dict

def predict_entities(model, tokenizer, label_list, image_path, words, device='cpu'):
    """
    Run NER inference on a single document image given its OCR words and bounding boxes.

    Args:
        model: RIFD model with an active ner_head
        tokenizer: TokenizerWrapper instance
        label_list: list of BIO label strings (e.g., ['B-QUESTION','I-QUESTION',...])
        image_path: path to the image (used only to normalise bounding boxes)
        words: list of dicts, each with 'text' (str) and 'box' (list [x0,y0,x1,y1] in pixels)
        device: torch device

    Returns:
        List of extracted entities, each with:
            'label'  – entity type (QUESTION, ANSWER, HEADER, OTHER)
            'text'   – concatenated words
            'box'    – merged bounding box [x0,y0,x1,y1] (pixel coords)
            'words'  – list of original word dicts belonging to this entity
    """
    model.eval()
    # Image dimensions for box normalisation
    img = cv2.imread(str(image_path))
    if img is None:
        raise ProjectException(f"Could not load image {image_path}")
    h, w = img.shape[:2]

    # Build input sequence with special tokens
    input_ids = [tokenizer.cls_token_id]
    bboxes = [[0.0, 0.0, 1.0, 1.0]]          # [CLS] box
    subword_map = []                           # subword index -> word index
    for word_idx, word in enumerate(words):
        box = word['box']
        norm_box = [box[0]/w, box[1]/h, box[2]/w, box[3]/h]
        sub_ids = tokenizer.tokenizer.encode(word['text'], add_special_tokens=False)
        for tid in sub_ids:
            input_ids.append(tid)
            bboxes.append(norm_box)
            subword_map.append(word_idx)
    input_ids.append(tokenizer.sep_token_id)
    bboxes.append([0.0, 0.0, 1.0, 1.0])

    # Convert to tensors (batch size 1)
    input_ids_t = torch.tensor([input_ids], dtype=torch.long).to(device)
    bboxes_t = torch.tensor([bboxes], dtype=torch.float).to(device)
    attn_mask_t = torch.ones_like(input_ids_t)

    with torch.no_grad():
        logits = model.forward_ner(input_ids_t, bboxes_t, attn_mask_t)  # (1, L, num_labels)
    preds = torch.argmax(logits, dim=-1)[0].cpu().tolist()  # (L)

    # Map subword predictions back to words (use the label of the first subword)
    word_labels = [None] * len(words)
    for i, pred_id in enumerate(preds):
        # skip [CLS] (i=0) and [SEP] (i=last)
        if i == 0 or i == len(preds) - 1:
            continue
        word_idx = subword_map[i - 1]          # -1 because we prepended [CLS]
        if word_labels[word_idx] is None:
            word_labels[word_idx] = pred_id

    # BIO decoding into entities
    entities = []
    current_entity = None
    for word_idx, (lbl_id, word) in enumerate(zip(word_labels, words)):
        if lbl_id is None:
            continue
        label_str = label_list[lbl_id]          # e.g., "B-QUESTION"
        if label_str.startswith('B-'):
            if current_entity:
                entities.append(current_entity)
            entity_type = label_str[2:]
            current_entity = {
                'label': entity_type,
                'text': word['text'],
                'box': word['box'][:],
                'words': [word]
            }
        elif label_str.startswith('I-'):
            entity_type = label_str[2:]
            if current_entity and current_entity['label'] == entity_type:
                current_entity['text'] += ' ' + word['text']
                # merge bounding boxes
                cb = current_entity['box']
                wb = word['box']
                current_entity['box'] = [
                    min(cb[0], wb[0]), min(cb[1], wb[1]),
                    max(cb[2], wb[2]), max(cb[3], wb[3])
                ]
                current_entity['words'].append(word)
            else:
                # orphan I- tag – start a new entity
                if current_entity:
                    entities.append(current_entity)
                current_entity = {
                    'label': entity_type,
                    'text': word['text'],
                    'box': word['box'][:],
                    'words': [word]
                }
        else:  # 'O'
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    if current_entity:
        entities.append(current_entity)

    return entities


# ---------- Demo on a FUNSD validation sample ----------
# Load best fine-tuned model (if available) or instantiate a new one for demonstration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Recreate model with same architecture and load weights
demo_model = RIFD(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=256,          # ← same as training
    num_heads=4,          # ←
    num_layers=4,         # ←
    d_ff=1024,            # ←
    max_len=512,
    dropout=0.1,
    num_ner_labels=FUNSDDataset.get_num_labels()
)
# Try loading best checkpoint
if Path('best-ner-model.pth').exists():
    demo_model.load_state_dict(torch.load('best-ner-model.pth', map_location=device))
    demo_model.to(device)
    print("Loaded fine-tuned weights from best-ner-model.pth")
else:
    print("WARNING: No fine-tuned checkpoint found; using randomly initialised model (results will be meaningless).")
    demo_model.to(device)

# Take one sample from validation set
val_dataset = FUNSDDataset(DATA_DIR / "funsd", tokenizer, max_length=512, split="test")

# Get a random index to select a sample
sample_idx = random.randint(0, len(val_dataset) - 1)

# The dataset returns tensors; we need the original words list for visualisation
# We'll re-read the annotation manually
img_path, ann_path = val_dataset.samples[sample_idx] # Use the integer index here
with open(ann_path) as f:
    annot = json.load(f)
words = []
for form_item in annot['form']:
    for w in form_item['words']:
        words.append({'text': w['text'], 'box': w['box']})

label_list = FUNSDDataset.get_label_list()
entities = predict_entities(demo_model, tokenizer, label_list, img_path, words, device)

print(f"\nExtracted entities from {img_path.name}:")
for ent in entities:
    print(f"  [{ent['label']}] \"{ent['text']}\"  box: {ent['box']}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Re-load the image to draw on it
img = cv2.imread(str(img_path))
if img is None:
    raise ProjectException(f"Failed to load image for visualization {img_path}")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(img)

# Define colors for different entity types if not already defined (from previous EDA cell)
if 'colors' not in locals():
    colors = {"question": "blue", "answer": "green", "header": "orange", "other": "gray"}

# Draw predicted entities
for ent in entities:
    box = ent['box']  # These are pixel coordinates [x0, y0, x1, y1]
    label = ent['label'].lower()
    color = colors.get(label, "red") # Default to red if label color not found

    # Create a Rectangle patch
    rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                             linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)

    # Add label text
    ax.text(box[0], box[1] - 10, ent['label'], fontsize=9, color=color,
            bbox=dict(facecolor='white', alpha=0.7))

ax.set_title(f"Detected Entities in {img_path.name}", fontsize=16)
ax.axis('off') # Hide axes ticks
plt.tight_layout()
plt.show()

In [ ]:
# Cell 21: Full test evaluation + ground‑truth vs prediction visualisation
import numpy as np
from seqeval.metrics import classification_report as seq_report
from tqdm import tqdm
import matplotlib.patches as patches

# ------------------------------
# 1. Load test dataset & model
# ------------------------------
test_dataset = FUNSDDataset(DATA_DIR / "funsd", tokenizer, max_length=512, split="test")
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = RIFD(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=256,          # ← same as training
    num_heads=4,          # ←
    num_layers=4,         # ←
    d_ff=1024,            # ←
    max_len=512,
    dropout=0.1,
    num_ner_labels=FUNSDDataset.get_num_labels()
)
if not Path('best-ner-model.pth').exists():
    raise ProjectException("best-ner-model.pth not found. Fine‑tune the model first.")
model.load_state_dict(torch.load('best-ner-model.pth', map_location=device))
model.to(device)
model.eval()

label_list = FUNSDDataset.get_label_list()

# ------------------------------
# 2. Collect predictions
# ------------------------------
all_labels = []   # list of token‑level label strings per document
all_preds = []    # same

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        bboxes = batch['bboxes'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model.forward_ner(input_ids, bboxes, attention_mask)
        preds = torch.argmax(logits, dim=-1)  # (B, L)

        for i in range(labels.size(0)):
            true = labels[i].cpu().tolist()
            pred = preds[i].cpu().tolist()
            true_seq, pred_seq = [], []
            for t, p in zip(true, pred):
                if t != FUNSDDataset.IGNORE_INDEX:
                    true_seq.append(label_list[t])
                    pred_seq.append(label_list[p])
            all_labels.append(true_seq)
            all_preds.append(pred_seq)

# ------------------------------
# 3. Token‑level accuracy
# ------------------------------
flat_true = [t for seq in all_labels for t in seq]
flat_pred = [p for seq in all_preds for p in seq]
token_acc = sum(1 for t, p in zip(flat_true, flat_pred) if t == p) / len(flat_true)
print(f"\nToken‑level accuracy: {token_acc:.4f}\n")

# ------------------------------
# 4. Entity‑level metrics (seqeval)
# ------------------------------
print("Entity‑level classification report:")
print(seq_report(all_labels, all_preds, digits=4))

# ------------------------------
# 5. Visualise random test samples: ground‑truth vs predicted entities
# ------------------------------
colors = {"QUESTION": "blue", "ANSWER": "green", "HEADER": "orange", "OTHER": "gray"}
num_vis = 3
vis_indices = np.random.choice(len(test_dataset), min(num_vis, len(test_dataset)), replace=False)

for idx in vis_indices:
    img_path, ann_path = test_dataset.samples[idx]

    # Load image & annotation
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    with open(ann_path, "r") as f:
        annot = json.load(f)

    # Build word list for our predictor
    words = []
    for form_item in annot["form"]:
        for w in form_item.get("words", []):
            words.append({"text": w["text"], "box": w["box"]})

    # Predicted entities
    ents_pred = predict_entities(model, tokenizer, label_list, img_path, words, device)

    # Ground‑truth entities (merged per field)
    ents_gt = []
    for form_item in annot["form"]:
        label = form_item.get("label", "other").upper()
        field_words = form_item.get("words", [])
        if not field_words:
            continue
        boxes = [w["box"] for w in field_words]
        merged_box = [
            min(b[0] for b in boxes),
            min(b[1] for b in boxes),
            max(b[2] for b in boxes),
            max(b[3] for b in boxes),
        ]
        text = " ".join(w["text"] for w in field_words)
        ents_gt.append({"label": label, "text": text, "box": merged_box})

    # Plot side by side
    fig, axes = plt.subplots(1, 2, figsize=(16, 10))
    for ax, ents, title in zip(axes, [ents_gt, ents_pred], ["Ground Truth", "Predicted"]):
        ax.imshow(img)
        ax.set_title(title, fontsize=14)
        ax.axis("off")
        for ent in ents:
            box = ent["box"]
            color = colors.get(ent["label"], "black")
            rect = patches.Rectangle(
                (box[0], box[1]), box[2]-box[0], box[3]-box[1],
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(box[0], box[1]-5, ent["label"], color=color, fontsize=8,
                    bbox=dict(facecolor='white', alpha=0.7))
    plt.suptitle(f"Document: {img_path.name}", fontsize=16)
    plt.tight_layout()
    plt.show()